In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
#Data load 
import pandas as pd

df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

#

print(df.head())

In [ ]:
'''
Rows: 7,043
Columns: 21
Target column: Churn
customerID → customer ki unique ID
Numerical columns → SeniorCitizen, tenure, MonthlyCharges, TotalCharges
Baaki kaafi columns categorical hain (Yes/No, Male/Female, contract types, etc.)
'''

In [ ]:
#data inspection print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

In [ ]:
#check Missing Values 
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
#check duplicates 
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
#describe  the data 
print(df.describe())

In [ ]:
#Categorical values check
for col in df.select_dtypes(include="object").columns:
    print("\n", col)
    print(df[col].value_counts())

In [ ]:
#Churn distribution
print(df["Churn"].value_counts())

In [ ]:
print(df["Churn"].value_counts(normalize=True) * 100)

In [ ]:
#EDA 
#step 1 Visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x="Churn", data=df)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
#Contract vs Churn
plt.figure(figsize=(8, 5))

sns.countplot(x="Contract", hue="Churn", data=df)

plt.title("Contract Type vs Churn")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
#Tenure vs Churn



plt.figure(figsize=(8, 5))

sns.histplot(data=df, x="tenure", hue="Churn", bins=30, kde=True)

plt.title("Tenure vs Churn")
plt.xlabel("Tenure (Months)")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
#Monthly Charges vs Churn

#Ab dekhte hain ki higher monthly charges wale customers mein churn zyada hai ya nahi.

plt.figure(figsize=(8, 5))

sns.boxplot(x="Churn", y="MonthlyCharges", data=df)

plt.title("Monthly Charges vs Churn")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.show()

In [ ]:
#Univariate Numerical Analysis



numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns

print(numerical_cols)

In [ ]:
df[numerical_cols].hist(figsize=(10, 6), bins=20)

plt.suptitle("Numerical Features Distribution")
plt.tight_layout()
plt.show()

In [ ]:
#Outliers 
plt.figure(figsize=(10, 5))

sns.boxplot(data=df[["tenure", "MonthlyCharges"]])

plt.title("Outlier Analysis")
plt.xlabel("Features")
plt.ylabel("Values")
plt.show()

In [ ]:
#Correlation Analysis / Heatmap 

#Ab hum numerical features ke beech relationship dekhenge.

correlation = df[["SeniorCitizen", "tenure", "MonthlyCharges"]].corr()

plt.figure(figsize=(8, 5))

sns.heatmap(correlation, annot=True, cmap="coolwarm")

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
'''Final EDA Insights 

Ab tak jo analysis kiya hai usse hum important observations note karenge:

Churn: 26.54% customers churn kar rahe hain.
Contract: Contract type aur churn ke beech noticeable relationship hai.
Tenure: Newer customers mein churn ka pattern important hai.
Monthly Charges: Charges aur churn ke distribution mein difference hai.
Numerical features: tenure, MonthlyCharges, SeniorCitizen analyze ho chuke hain.
Data quality: duplicates nahi mile; lekin TotalCharges ke blank strings ko cleaning phase mein specifically check karna hai.'''

In [ ]:
#Data Cleaning
#TotalCharges check


print("Blank TotalCharges:",
      df["TotalCharges"].astype(str).str.strip().eq("").sum())

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print(df["TotalCharges"].isnull().sum())

In [ ]:
df = df.dropna(subset=["TotalCharges"])

print("Shape after cleaning:", df.shape)
print("Missing TotalCharges:", df["TotalCharges"].isnull().sum())

In [ ]:
#Check dataTypes
print(df.dtypes)

In [ ]:
#Clean customerIDs
df = df.drop("customerID", axis=1)

print(df.shape)
print(df.columns)

In [ ]:
#Target Encoding



#Model ko numerical values chahiye, isliye No → 0 aur Yes → 1 karenge.

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print(df["Churn"].value_counts())
print(df["Churn"].dtype)

In [ ]:
#Feature Encoding

'''Ab problem ye hai ki hamare bahut saare features text/categorical hain, jaise:

gender, Partner, Contract, InternetService, PaymentMethod etc.

ML model directly in text values ko understand nahi karega.

Isliye next hum categorical features ko numerical format mein convert karenge using One-Hot Encoding.

Pehle identify karte hain ki kitne categorical columns bache hain:'''

categorical_cols = df.select_dtypes(include=["object", "str"]).columns

print(categorical_cols)
print("Total categorical columns:", len(categorical_cols))

In [ ]:
#Seperate X and Y 
X = df.drop("Churn", axis=1)
y = df["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:


#We were at Train/Test Split. Since you've already separated X and y, run this:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

'''Why stratify=y?

Our target has:

73.46% → No Churn
26.54% → Churn

stratify=y keeps approximately the same churn ratio in both training and testing data.'''

In [ ]:
'''One-Hot Encoding

Now we need to convert the 15 categorical columns into numerical columns.

We'll use OneHotEncoder through a ColumnTransformer. This is better than manually encoding because it keeps the preprocessing organized and prevents data leakage.
'''

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_cols = X_train.select_dtypes(include=["object", "str"]).columns
numerical_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("X_train encoded shape:", X_train_encoded.shape)
print("X_test encoded shape:", X_test_encoded.shape)

#We fit only on training data. This is an important ML practice to avoid data leakage.

In [ ]:
#Next — Check the encoded data

#Before training a model, let's verify the final feature size:

print("Original features:", X_train.shape[1])
print("Encoded features:", X_train_encoded.shape[1])

In [ ]:
#It means your current X_train has 45 columns/features.

#But earlier we expected 19 from the original dataset, so something has likely changed in your notebook. Before moving to model training, let's verify exactly what is inside X_train.
print(X_train.shape)
print(X_train.columns.tolist())
#Original data → 19 features → One-Hot Encoding → 48 features

In [ ]:
#Model Train Parts
#Logistic Regression

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=5000)

model.fit(X_train_encoded, y_train)

y_pred = model.predict(X_test_encoded)

print(y_pred[:10])

In [ ]:
#Logistic Regression evaluation

'''
Accuracy
Confusion Matrix
Precision / Recall / F1
ROC-AUC
'''

In [ ]:
#Accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

In [ ]:
#classification report
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
#confusion Matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

In [ ]:
#explanation 
'''Isko hum 4 parts mein read karte hain:

Actual	Predicted	        Count	      Meaning
0	        0	            915	          Correct  No Churn → TN
0	 	    1               118	          Wrongly predicted Churn → FP
1		    0               158	          Churn customer miss ho gaya → FN
1		    1               216	          Correctly predicted Churn → TP'''

In [ ]:

'''
Ab Logistic Regression ka conclusion

Accuracy = 80%
Churn Recall = 58%
Churn F1 = 61%
'''

In [ ]:
#Step — ROC-AUC
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test_encoded)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", roc_auc)

#like [:, 1] ka matlab hai Churn = 1 ki probability.


#explanation 
'''Ye Logistic Regression ke liye strong baseline hai.

ROC-AUC ka simple meaning

ROC-AUC basically check karta hai ki model Churn (1) aur No Churn (0) customers ko kitna achhe se distinguish kar pa raha hai.

0.5 -> random guessing
0.6 ->0.7 → weak
0.7 ->0.8 → decent
0.8 ->0.9 → good
0.9+ → excellent'''


#conclusion of LogisticRegression
'''Accuracy   = 80%
Precision  = 65%  (Churn)
Recall     = 58%  (Churn)
F1-score   = 61%  (Churn)
ROC-AUC    = 0.8358'''

In [ ]:
#KNN 

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor_knn = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numerical_cols)
    ]
)

X_train_knn = preprocessor_knn.fit_transform(X_train)
X_test_knn = preprocessor_knn.transform(X_test)

print("Training shape:", X_train_knn.shape)
print("Testing shape:", X_test_knn.shape)

#Scaling + preprocessing successfully done.

In [ ]:
# KNN model, k=5
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_knn, y_train)

y_pred_knn = knn_model.predict(X_test_knn)

print(y_pred_knn[:10])

In [ ]:
#Accuracy and Classification
from sklearn.metrics import accuracy_score, classification_report

accuracy_knn = accuracy_score(y_test, y_pred_knn)

print("KNN Accuracy:", accuracy_knn)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

In [ ]:
#confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm_knn = confusion_matrix(y_test, y_pred_knn)

print(cm_knn)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_knn,
    display_labels=["No Churn", "Churn"]
)

disp.plot()
plt.show()

'''
TN = 854 → correctly predicted No Churn
FP = 179 → No Churn ko Churn predict kiya
FN = 156 → Churn customer ko No Churn predict kiya ⚠️
TP = 218 → correctly predicted Churn

Churn recall = 218 / (218 + 156) ≈ 58%, jo classification report se match karta hai. 
'''

In [ ]:
#ROC - AUC
from sklearn.metrics import roc_auc_score

y_prob_knn = knn_model.predict_proba(X_test_knn)[:, 1]

roc_auc_knn = roc_auc_score(y_test, y_prob_knn)

print("KNN ROC-AUC:", roc_auc_knn)

In [ ]:
#quick summary


#KNN Final Result

'''Accuracy: 76.19%
Churn Recall: 58%
Churn F1: 57%
ROC-AUC: ~0.77'''


#🆚 Current Comparison
'''
| Model               | Accuracy | Churn Recall |      F1 |   ROC-AUC |
| ------------------- | -------: | -----------: | ------: | --------: |
| Logistic Regression |  **80%** |          58% | **61%** | **0.836** |
| KNN                 |   76.19% |          58% |     57% |     ~0.77 |
'''

#So KNN is currently weaker than Logistic Regression.

In [ ]:
#KNN tuning with different K values

In [ ]:
k_values = [3, 5, 7, 9, 11]

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_knn, y_train)

    y_pred = knn.predict(X_test_knn)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"K = {k}, Accuracy = {accuracy:.4f}")


'''
Observation
K=3 → 75.62%
K=5 → 76.19%
K=7 → 76.05%
K=9 → 77.26% ⭐ Best
K=11 → 76.83%
'''

In [ ]:
#Final KNN Model
knn_model = KNeighborsClassifier(n_neighbors=9)

knn_model.fit(X_train_knn, y_train)

y_pred_knn = knn_model.predict(X_test_knn)

print("Final KNN Accuracy:",
      accuracy_score(y_test, y_pred_knn))

In [ ]:
# Implementation of Decision TREE

'''
                 Contract?
                /         \
          Month-to-month   2 year
              /              \
        High Charges?       No Churn
          /    \
        Yes     No
       Churn   No Churn
'''

'''
Important terms:

Root → first decision
Node → decision/question
Branch → decision ka outcome
Leaf → final prediction
Gini Impurity → split ki purity measure karta hai

'''

In [ ]:
#Decision Tree Modelfrom sklearn.tree import DecisionTreeClassifier

from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    random_state=42
)

dt_model.fit(X_train_encoded, y_train)

y_pred_dt = dt_model.predict(X_test_encoded)

print(y_pred_dt[:10])

In [ ]:
#Accuracy + Classification Report:

from sklearn.metrics import accuracy_score, classification_report

accuracy_dt = accuracy_score(y_test, y_pred_dt)

print("Decision Tree Accuracy:", accuracy_dt)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))


'''
Result
Accuracy: 72.21%
Churn Precision: 48%
Churn Recall: 49%
Churn F1: 49%'''

In [ ]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm_dt = confusion_matrix(y_test, y_pred_dt)

print(cm_dt)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_dt,
    display_labels=["No Churn", "Churn"]
)

disp.plot()
plt.show()

In [ ]:
#ROC- AUC
from sklearn.metrics import roc_auc_score

y_prob_dt = dt_model.predict_proba(X_test_encoded)[:, 1]

roc_auc_dt = roc_auc_score(y_test, y_prob_dt)

print("Decision Tree ROC-AUC:", roc_auc_dt)

In [ ]:
#comparision
'''
| Model               |   Accuracy | Churn Recall | Churn F1 |   ROC-AUC |
| ------------------- | ---------: | -----------: | -------: | --------: |
| Logistic Regression | **80.00%** |          58% |  **61%** | **0.836** |
| KNN (K=9)           |     77.26% |          58% |      57% |     ~0.77 |
| Decision Tree       |     72.21% |          49% |      49% |  **0.64** |
'''

In [ ]:
#max_depth experiment
depths = [3, 5, 7, 10, None]

for depth in depths:
    dt = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    dt.fit(X_train_encoded, y_train)

    y_pred = dt.predict(X_test_encoded)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"max_depth = {depth}, Accuracy = {accuracy:.4f}")

In [ ]:
#result
'''
| Max Depth |      Accuracy |
| --------: | ------------: |
|         3 |        77.83% |
|     **5** | **78.96% 🏆** |
|         7 |        77.75% |
|        10 |        75.55% |
|      None |        72.21% |

'''

In [ ]:
'''
What did we learn?
max_depth=None → tree bahut deep gaya → overfitting ka risk → 72.21%
Depth increase karte hue performance pehle improve hui.
Depth = 5 par best accuracy: 78.96%
Uske baad depth badhne par accuracy decrease hone lagi.

So, abhi ke experiment ke according:

Best Decision Tree depth = 5'''

In [ ]:
#Final Decision Tree

#then train with the value of  max_depth=5 .

dt_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt_model.fit(X_train_encoded, y_train)

y_pred_dt = dt_model.predict(X_test_encoded)

print("Final Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))

In [ ]:
#classificatin report
from sklearn.metrics import classification_report

print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))

In [ ]:
'''
Decision Tree — Final Interpretation
Accuracy: 78.96%
Class 0 (No Churn):
Precision: 86%
Recall: 85%
F1: 86%
Class 1 (Churn):
Precision: 60%
Recall: 61%
F1: 61%
'''

In [ ]:
#confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm_dt = confusion_matrix(y_test, y_pred_dt)

print(cm_dt)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_dt,
    display_labels=["No Churn", "Churn"]
)

disp.plot()
plt.show()


'''
Meaning:

TN = 881 → actual No Churn, predicted No Churn
FP = 152 → actual No Churn, predicted Churn
FN = 144 → actual Churn, predicted No Churn
TP = 230 → actual Churn, predicted Churn'''

In [ ]:
#ROC-AUC at maxdepth =5
from sklearn.metrics import roc_auc_score

y_prob_dt = dt_model.predict_proba(X_test_encoded)[:, 1]

roc_auc_dt = roc_auc_score(y_test, y_prob_dt)

print("Decision Tree ROC-AUC:", roc_auc_dt)

In [ ]:
'''
Decision  Tree final result
| Metric          |     Result |
| --------------- | ---------: |
| Max Depth       |      **5** |
| Accuracy        | **78.96%** |
| Churn Precision |        60% |
| Churn Recall    |    **61%** |
| Churn F1        |    **61%** |
| ROC-AUC         |  **0.830** |
'''

In [ ]:
#Final comparision
'''
| Model                   |   Accuracy | Churn Recall | Churn F1 |   ROC-AUC |
| ----------------------- | ---------: | -----------: | -------: | --------: |
| Logistic Regression     | **80.00%** |          58% |  **61%** | **0.836** |
| KNN (K=9)               |     77.26% |          58% |      57% |     ~0.77 |
| Decision Tree (depth=5) |     78.96% |      **61%** |  **61%** | **0.830** |
'''


In [ ]:
#Random Forest Concept

#Random Forest = collection of multiple Decision Trees.

'''Ek single Decision Tree prediction deta hai:

Decision Tree 1 → Churn'''
#In random Forest
'''
Tree 1 → Churn
Tree 2 → No Churn
Tree 3 → Churn
Tree 4 → Churn
Tree 5 → Churn

        ↓

Majority Voting

        ↓

      Churn
'''



'''
Important parameter

n_estimators = forest mein kitne decision trees honge.

For example:

RandomForestClassifier(n_estimators=100)

means approximately 100 trees.
'''


#Model Train

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_encoded, y_train)

y_pred_rf = rf_model.predict(X_test_encoded)

print(y_pred_rf[:10])

In [ ]:
#Accuracy + Classification
from sklearn.metrics import accuracy_score, classification_report

accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", accuracy_rf)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))



'''
Random Forest — Baseline
Accuracy: 79.25%
Class 0 recall: 90%
Class 1 (Churn) recall: 51%
Churn F1: 57%'''

In [ ]:
#confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm_rf = confusion_matrix(y_test, y_pred_rf)

print(cm_rf)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=["No Churn", "Churn"]
)

disp.plot()
plt.show()



'''
TN = 925 → correctly predicted No Churn
FP = 108 → No Churn predicted as Churn
FN = 184 → Churn predicted as No Churn
TP = 190 → correctly predicted Churn
'''

In [ ]:
#ROC- AUC
from sklearn.metrics import roc_auc_score

y_prob_rf = rf_model.predict_proba(X_test_encoded)[:, 1]

roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

print("Random Forest ROC-AUC:", roc_auc_rf)

In [ ]:
#comparision
'''
| Model                   |   Accuracy | Churn Recall | Churn F1 | ROC-AUC |
| ----------------------- | ---------: | -----------: | -------: | ------: |
| Logistic Regression     |     80.00% |          58% |      61% |   0.836 |
| KNN (K=9)               |     77.26% |          58% |      57% |   ~0.77 |
| Decision Tree (depth=5) |     78.96% |      **61%** |  **61%** |   0.830 |
| Random Forest           | **79.25%** |          51% |      57% |   0.810 |

'''

In [ ]:
# n _ estimators experiment
estimators = [50, 100, 200]

for n in estimators:
    rf = RandomForestClassifier(
        n_estimators=n,
        random_state=42
    )

    rf.fit(X_train_encoded, y_train)

    y_pred = rf.predict(X_test_encoded)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"n_estimators = {n}, Accuracy = {accuracy:.4f}")


    #result
    '''
|   Trees |   Accuracy |
| ------: | ---------: |
|      50 |     78.89% |
| **100** | **79.25%** |
|     200 |     78.68% |

    '''

In [ ]:
#Final random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_encoded, y_train)

y_pred_rf = rf_model.predict(X_test_encoded)

print("Final Random Forest Accuracy:",
      accuracy_score(y_test, y_pred_rf))

In [ ]:
#final ROC-AUC

from sklearn.metrics import roc_auc_score

y_prob_rf = rf_model.predict_proba(X_test_encoded)[:, 1]

roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

print("Final Random Forest ROC-AUC:", roc_auc_rf)



#Result
'''
| Metric          | Random Forest |
| --------------- | ------------: |
| `n_estimators`  |           100 |
| Accuracy        |    **79.25%** |
| Churn Precision |           64% |
| Churn Recall    |           51% |
| Churn F1        |           57% |
| ROC-AUC         |      **0.81** |
'''

In [ ]:
#comparison
'''
| Model                   |   Accuracy | Churn Recall | Churn F1 |   ROC-AUC |
| ----------------------- | ---------: | -----------: | -------: | --------: |
| Logistic Regression     | **80.00%** |          58% |      61% | **0.836** |
| KNN (K=9)               |     77.26% |          58% |      57% |     ~0.77 |
| Decision Tree (depth=5) |     78.96% |      **61%** |  **61%** |     0.830 |
| Random Forest (100)     |     79.25% |          51% |      57% |     0.810 |
'''

In [ ]:
'''
Part 1 — Naive Bayes
Concept + intuition
Categorical/Numeric data ke context mein samajhna
Model train
Prediction
Accuracy
Classification Report
Confusion Matrix
ROC-AUC
Part 2 — SVM
SVM concept
Hyperplane + margin
C aur kernel ka basic idea
Training
Prediction
Accuracy
Classification Report
Confusion Matrix
ROC-AUC
Part 3
Naive Bayes vs SVM vs previous models
Basic comparison
'''

In [ ]:
print(type(X_train_encoded))
print(X_train_encoded.shape)

In [ ]:
#Train Naive Byes
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()

nb_model.fit(X_train_encoded, y_train)

y_pred_nb = nb_model.predict(X_test_encoded)

print(y_pred_nb[:10])

In [ ]:
#Accuracy + Classification Report
from sklearn.metrics import accuracy_score, classification_report

accuracy_nb = accuracy_score(y_test, y_pred_nb)

print("Naive Bayes Accuracy:", accuracy_nb)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))




In [ ]:
'''Naive Bayes Result
Accuracy: 68.23%
Churn Precision: 45%
Churn Recall: 83%
Churn F1: 58%'''

In [ ]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm_nb = confusion_matrix(y_test, y_pred_nb)

print(cm_nb)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_nb,
    display_labels=["No Churn", "Churn"]
)

disp.plot()
plt.show()

In [ ]:
'''So:

TN = 650
FP = 383
FN = 64
TP = 310
'''

In [ ]:
#ROC - AUC
from sklearn.metrics import roc_auc_score

y_prob_nb = nb_model.predict_proba(X_test_encoded)[:, 1]

roc_auc_nb = roc_auc_score(y_test, y_prob_nb)

print("Naive Bayes ROC-AUC:", roc_auc_nb)

In [ ]:
'''
Naive Byes Result 
| Metric          |     Result |
| --------------- | ---------: |
| Accuracy        | **68.23%** |
| Churn Precision |    **45%** |
| Churn Recall    |    **83%** |
| Churn F1        |    **58%** |
| ROC-AUC         | **0.8053** |

'''

In [ ]:
#SVM 
#SVM Train + Predict
from sklearn.svm import SVC

svm_model = SVC(
    kernel="rbf",
    C=1,
    probability=True,
    random_state=42
)

svm_model.fit(X_train_knn, y_train)

y_pred_svm = svm_model.predict(X_test_knn)

print(y_pred_svm[:10])

In [ ]:
#Classification Report and Accuracy
from sklearn.metrics import accuracy_score, classification_report

accuracy_svm = accuracy_score(y_test, y_pred_svm)

print("SVM Accuracy:", accuracy_svm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

In [ ]:
#SVM result
'''
| Metric          |        SVM |
| --------------- | ---------: |
| Accuracy        | **79.18%** |
| Churn Precision |    **64%** |
| Churn Recall    |    **49%** |
| Churn F1        |    **56%** |
'''

In [ ]:
#confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm_svm = confusion_matrix(y_test, y_pred_svm)

print(cm_svm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_svm,
    display_labels=["No Churn", "Churn"]
)

disp.plot()
plt.show()

In [ ]:
'''TN = 929 → actual No Churn, predicted No Churn
FP = 104 → actual No Churn, predicted Churn
FN = 189 → actual Churn, predicted No Churn
TP = 185 → actual Churn, predicted Churn'''

In [ ]:
#SVM ROC - AUC
from sklearn.metrics import roc_auc_score

y_prob_svm = svm_model.predict_proba(X_test_knn)[:, 1]

roc_auc_svm = roc_auc_score(y_test, y_prob_svm)

print("SVM ROC-AUC:", roc_auc_svm)

In [ ]:
#SVM result
'''
| Metric          |        SVM |
| --------------- | ---------: |
| Accuracy        | **79.18%** |
| Churn Precision |    **64%** |
| Churn Recall    |    **49%** |
| Churn F1        |    **56%** |
| ROC-AUC         |  **~0.78** |
'''

In [ ]:
#Overcall result
'''
| Model                   |   Accuracy | Churn Recall | Churn F1 |   ROC-AUC |
| ----------------------- | ---------: | -----------: | -------: | --------: |
| Logistic Regression     |     80.00% |          58% |      61% |     0.836 |
| KNN (K=9)               |     77.26% |          58% |      57% |     ~0.77 |
| Decision Tree (Depth=5) |     78.96% |          61% |      61% |     0.830 |
| Random Forest           |     79.25% |          51% |      57% |     0.810 |
| **Naive Bayes**         | **68.23%** |      **83%** |  **58%** | **0.805** |
| **SVM**                 | **79.18%** |      **49%** |  **56%** | **~0.78** |

'''

In [ ]:
#Model Comparison & Evaluation.

In [ ]:
#Comparison DataFrame
import pandas as pd

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "KNN",
        "Decision Tree",
        "Random Forest",
        "Naive Bayes",
        "SVM"
    ],
    "Accuracy": [
        0.8000,
        0.7726,
        0.7896,
        0.7925,
        0.6823,
        0.7918
    ],
    "Churn Recall": [
        0.58,
        0.58,
        0.61,
        0.51,
        0.83,
        0.49
    ],
    "Churn F1": [
        0.61,
        0.57,
        0.61,
        0.57,
        0.58,
        0.56
    ],
    "ROC-AUC": [
        0.8358,
        0.77,
        0.8296,
        0.81,
        0.8053,
        0.78
    ]
})

model_comparison

In [ ]:
#percentage conversion
comparison_display = model_comparison.copy()

metrics = ["Accuracy", "Churn Recall", "Churn F1", "ROC-AUC"]

comparison_display[metrics] = comparison_display[metrics] * 100

comparison_display[metrics] = comparison_display[metrics].round(2)

comparison_display

In [ ]:
#Accuracy Comparison
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.bar(
    comparison_display["Model"],
    comparison_display["Accuracy"]
)

plt.xlabel("Model")
plt.ylabel("Accuracy (%)")
plt.title("Model Accuracy Comparison")

plt.xticks(rotation=30)
plt.ylim(0, 100)

plt.show()

In [ ]:
#Chrun recall comparison
plt.figure(figsize=(10, 5))

plt.bar(
    comparison_display["Model"],
    comparison_display["Churn Recall"]
)

plt.xlabel("Model")
plt.ylabel("Churn Recall (%)")
plt.title("Churn Recall Comparison")

plt.xticks(rotation=30)
plt.ylim(0, 100)

plt.show()

In [ ]:
#F1 comparison
plt.figure(figsize=(10, 5))

plt.bar(
    comparison_display["Model"],
    comparison_display["Churn F1"]
)

plt.xlabel("Model")
plt.ylabel("Churn F1 (%)")
plt.title("Churn F1-Score Comparison")

plt.xticks(rotation=30)
plt.ylim(0, 100)

plt.show()

In [ ]:
# ROC-AUC Comparison

plt.figure(figsize=(10, 5))

plt.bar(
    comparison_display["Model"],
    comparison_display["ROC-AUC"]
)

plt.xlabel("Model")
plt.ylabel("ROC-AUC (%)")
plt.title("ROC-AUC Comparison")

plt.xticks(rotation=30)
plt.ylim(0, 100)

plt.show()

In [ ]:
#Model Behavior Analysis
'''
| Model               | Accuracy | Churn Recall | Churn F1 | ROC-AUC |
| ------------------- | -------: | -----------: | -------: | ------: |
| Logistic Regression |   80.00% |          58% |      61% |  83.58% |
| KNN                 |   77.26% |          58% |      57% |  77.00% |
| Decision Tree       |   78.96% |          61% |      61% |  82.96% |
| Random Forest       |   79.25% |          51% |      57% |  81.00% |
| Naive Bayes         |   68.23% |      **83%** |      58% |  80.53% |
| SVM                 |   79.18% |          49% |      56% |  78.00% |

'''

'''
1. Logistic Regression

Accuracy = 80%
ROC-AUC = 83.58%
Churn F1 = 61%

Balanced performance de raha hai.

2. KNN

Accuracy = 77.26%
ROC-AUC = 77%
Churn F1 = 57%

Overall metrics comparatively lower hain.

3. Decision Tree

Accuracy = 78.96%
Churn Recall = 61%
Churn F1 = 61%
ROC-AUC = 82.96%

Churn class ko Logistic Regression ke comparison mein thoda zyada recall karta hai.

4. Random Forest

Accuracy = 79.25%
Churn Recall = 51%
ROC-AUC = 81%

Overall accuracy reasonable hai, but current configuration mein churn recall lower hai.

5. Naive Bayes

Accuracy = 68.23%
Churn Recall = 83%
ROC-AUC = 80.53%

Yahan interesting trade-off hai: bahut saare actual churners detect ho rahe hain, but false positives bhi kaafi hain.

6. SVM

Accuracy = 79.18%
Churn Recall = 49%
ROC-AUC = 78%

Current configuration mein churn recall relatively low hai.
'''

#Highest accuracy ≠ automatically best model.


In [ ]:
import os

print(os.getcwd())

In [ ]:
comparison_display.to_csv(
    "model_comparison.csv",
    index=False
)

print("Model comparison saved successfully!")

In [ ]:
#Feature Engineering
#current dataset check

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

In [ ]:
#TenureGroup Feature

'''Create TenureGroup
We are converting the numerical tenure feature into customer lifecycle groups.
Why?
tenure = 8 and tenure = 10 are both relatively new customers, while tenure = 60 represents a long-term customer.
A grouped feature can help the model capture these lifecycle patterns.'''

df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, float("inf")],
    labels=["New", "Early", "Mid", "Long-term"]
)

print(df["TenureGroup"].value_counts().sort_index())


'''What we learned
There are 7,032 customers divided into:
- New: 2,175
- Early: 1,024
- Mid: 1,594
- Long-term: 2,239'''

In [ ]:
#Step 3 — Average Monthly Spending
'''We already have:
- TotalCharges → total amount paid by customer
- tenure → number of months
So we can create:
AverageMonthlySpend = TotalCharges / tenure'''


df["AvgMonthlySpend"] = df["TotalCharges"] / df["tenure"].replace(0, 1)

print(df[["tenure", "TotalCharges", "AvgMonthlySpend"]].head())

In [ ]:
'''
Step 4 — Service Count
Ab hum customer ke paas kitni additional services hain, uska ek combined feature banayenge.
We'll count these Yes services:
- OnlineSecurity
- OnlineBackup
- DeviceProtection
- TechSupport
- StreamingTV
- StreamingMovies
'''
service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["ServiceCount"] = (df[service_cols] == "Yes").sum(axis=1)

print(df[service_cols + ["ServiceCount"]].head())


In [ ]:
#Contract-based feature
'''
IsMonthToMonth
- Month-to-month → 1
- Otherwise → 0
'''
df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

print(df[["Contract", "IsMonthToMonth"]].head(10))
print("\nValue Counts:")
print(df["IsMonthToMonth"].value_counts())

In [ ]:
#Total Feature Check
engineered_features = [
    "TenureGroup",
    "AvgMonthlySpend",
    "ServiceCount",
    "IsMonthToMonth"
]

print(df[engineered_features].isnull().sum())

In [ ]:
#current status
'''
| Feature | Purpose |
|---|---|
| `TenureGroup` | Customer tenure category |
| `AvgMonthlySpend` | Average monthly spending |
| `ServiceCount` | Number of subscribed services |
| `IsMonthToMonth` | Identifies month-to-month contracts |
'''

In [ ]:
'''Create New Feature-Engineered X and y'''

X_fe = df.drop("Churn", axis=1)
y_fe = df["Churn"]

print("X shape:", X_fe.shape)
print("y shape:", y_fe.shape)

In [ ]:
# Test/Train Split
from sklearn.model_selection import train_test_split

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe,
    y_fe,
    test_size=0.2,
    random_state=42,
    stratify=y_fe
)

print("X_train:", X_train_fe.shape)
print("X_test:", X_test_fe.shape)
print("y_train:", y_train_fe.shape)
print("y_test:", y_test_fe.shape)

In [ ]:
#Identify Categorical & Numerical Features
categorical_cols_fe = X_train_fe.select_dtypes(
    include=["object", "category", "str"]
).columns

numerical_cols_fe = X_train_fe.select_dtypes(
    include=["int64", "float64"]
).columns

print("Categorical Features:")
print(categorical_cols_fe.tolist())

print("\nNumerical Features:")
print(numerical_cols_fe.tolist())

In [ ]:
#Create Feature-Engineering Preprocessor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor_fe = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols_fe
        ),
        (
            "num",
            "passthrough",
            numerical_cols_fe
        )
    ]
)

X_train_fe_encoded = preprocessor_fe.fit_transform(X_train_fe)
X_test_fe_encoded = preprocessor_fe.transform(X_test_fe)

print("Encoded X_train shape:", X_train_fe_encoded.shape)
print("Encoded X_test shape:", X_test_fe_encoded.shape)

In [ ]:
#Train Feature-Engineered Logistic Regression

'''Before Feature Engineering: Accuracy = 80.00%, ROC-AUC = 83.58%
vs.
After Feature Engineering: new results.'''

from sklearn.linear_model import LogisticRegression

lr_fe = LogisticRegression(max_iter=5000)

lr_fe.fit(X_train_fe_encoded, y_train_fe)

y_pred_lr_fe = lr_fe.predict(X_test_fe_encoded)

print("Feature-Engineered Logistic Regression Accuracy:",
      lr_fe.score(X_test_fe_encoded, y_test_fe))

In [ ]:
'''Before Feature Engineering
- Accuracy = 80.00%
After Feature Engineering
- Accuracy = 79.25%'''

In [ ]:
#Complete Logistic Regression Evaluation

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test_fe, y_pred_lr_fe))

print("\nClassification Report:")
print(classification_report(y_test_fe, y_pred_lr_fe))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_fe, y_pred_lr_fe))

y_prob_lr_fe = lr_fe.predict_proba(X_test_fe_encoded)[:, 1]

print("\nROC-AUC:", roc_auc_score(y_test_fe, y_prob_lr_fe))

In [ ]:
#Logistic Regression — Before vs After Feature Engineering
'''
| Metric       | Before FE | After FE | Change    |
|--------------|-----------|----------|-----------|
| Accuracy     | 80.00%    | 79.25%   | ↓ 0.75 pp |
| Churn Recall | 58%       | 52%      | ↓ 6 pp    |
| Churn F1     | 61%       | 57%      | ↓ 4 pp    |
| ROC-AUC      | 83.58%    | 83.49%   | ↓ 0.09 pp |
'''

In [ ]:
'''Feature-Engineered Decision Tree
Same best setting use karenge: max_depth=5.'''

from sklearn.tree import DecisionTreeClassifier

dt_fe = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt_fe.fit(X_train_fe_encoded, y_train_fe)

y_pred_dt_fe = dt_fe.predict(X_test_fe_encoded)

print(
    "Feature-Engineered Decision Tree Accuracy:",
    dt_fe.score(X_test_fe_encoded, y_test_fe)
)

In [ ]:
'''✅ Decision Tree mein bhi accuracy improve nahi hui.
- Before FE: 78.96%
- After FE: 78.39%
- Change: −0.57 percentage points'''

In [ ]:
'''Feature-Engineered Random Forest
Hum wahi baseline configuration use karenge: 100 trees.'''
from sklearn.ensemble import RandomForestClassifier

rf_fe = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_fe.fit(X_train_fe_encoded, y_train_fe)

y_pred_rf_fe = rf_fe.predict(X_test_fe_encoded)

print(
    "Feature-Engineered Random Forest Accuracy:",
    rf_fe.score(X_test_fe_encoded, y_test_fe)
)

In [ ]:
#Feature Engineering — Current Results
'''
| Model               | Before FE | After FE | Change    |
|---------------------|-----------|----------|--------   |
| Logistic Regression | 80.00%    | 79.25%   | ↓ 0.75 pp |
| Decision Tree       | 78.96%    | 78.39%   | ↓ 0.57 pp |
| Random Forest       | 79.25%    | 78.68%   | ↓ 0.57 pp |
'''

In [ ]:
#Detailed Random Forest Evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test_fe, y_pred_rf_fe))

print("\nClassification Report:")
print(classification_report(y_test_fe, y_pred_rf_fe))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_fe, y_pred_rf_fe))

y_prob_rf_fe = rf_fe.predict_proba(X_test_fe_encoded)[:, 1]

print("\nROC-AUC:", roc_auc_score(y_test_fe, y_prob_rf_fe))

In [ ]:
'''
Random Forest — Before vs After FE
| Metric       | Before FE | After FE | Change        |
|--------------|-----------|----------|---------------|
| Accuracy     | 79.25%    | 78.68%   | ↓ 0.57 pp     |
| Churn Recall | 51%       | 49%      | ↓ 2 pp        |
| Churn F1     | 57%       | 55%      | ↓ 2 pp        |
| ROC-AUC      | 81.00%    | 81.48%   | **↑ 0.48 pp** |
'''

In [ ]:
#Create Clean Feature Set
features_to_drop = [
    "AvgMonthlySpend",
    "IsMonthToMonth"
]

X_fe_clean = df.drop(
    columns=["Churn"] + features_to_drop
)

y_fe_clean = df["Churn"]

print("X shape:", X_fe_clean.shape)
print("Features:")
print(X_fe_clean.columns.tolist())

In [ ]:
'''
Rebuild Train/Test Split
Same 80/20 + stratify + random_state=42 maintain karenge:
'''
X_train_fe_clean, X_test_fe_clean, y_train_fe_clean, y_test_fe_clean = train_test_split(
    X_fe_clean,
    y_fe_clean,
    test_size=0.2,
    random_state=42,
    stratify=y_fe_clean
)

print("X_train:", X_train_fe_clean.shape)
print("X_test:", X_test_fe_clean.shape)
print("y_train:", y_train_fe_clean.shape)
print("y_test:", y_test_fe_clean.shape)

In [ ]:
#identify features
categorical_cols_fe_clean = X_train_fe_clean.select_dtypes(
    include=["object", "category", "str"]
).columns

numerical_cols_fe_clean = X_train_fe_clean.select_dtypes(
    include=["int64", "float64"]
).columns

print("Categorical Features:")
print(categorical_cols_fe_clean.tolist())

print("\nNumerical Features:")
print(numerical_cols_fe_clean.tolist())

In [ ]:
#preprocessing
preprocessor_fe_clean = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols_fe_clean
        ),
        (
            "num",
            "passthrough",
            numerical_cols_fe_clean
        )
    ]
)

X_train_fe_clean_encoded = preprocessor_fe_clean.fit_transform(
    X_train_fe_clean
)

X_test_fe_clean_encoded = preprocessor_fe_clean.transform(
    X_test_fe_clean
)

print(
    "Encoded X_train shape:",
    X_train_fe_clean_encoded.shape
)

print(
    "Encoded X_test shape:",
    X_test_fe_clean_encoded.shape
)

In [ ]:
#Clean Feature-Engineered Logistic Regression
lr_fe_clean = LogisticRegression(max_iter=5000)

lr_fe_clean.fit(
    X_train_fe_clean_encoded,
    y_train_fe_clean
)

y_pred_lr_fe_clean = lr_fe_clean.predict(
    X_test_fe_clean_encoded
)

print(
    "Clean Feature-Engineered Logistic Regression Accuracy:",
    lr_fe_clean.score(
        X_test_fe_clean_encoded,
        y_test_fe_clean
    )
)

In [ ]:
'''
 interesting result .
Original Logistic Regression: 80.00%
Clean FE Logistic Regression: 79.32%
Difference = −0.68 percentage points.
'''

In [ ]:
#Complete Evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("Accuracy:", accuracy_score(
    y_test_fe_clean,
    y_pred_lr_fe_clean
))

print("\nClassification Report:")
print(classification_report(
    y_test_fe_clean,
    y_pred_lr_fe_clean
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_fe_clean,
    y_pred_lr_fe_clean
))

y_prob_lr_fe_clean = lr_fe_clean.predict_proba(
    X_test_fe_clean_encoded
)[:, 1]

print("\nROC-AUC:", roc_auc_score(
    y_test_fe_clean,
    y_prob_lr_fe_clean
))

In [ ]:
#Final Comparison
'''
| Metric       | Original LR | Clean FE LR | Change   |
|--------------|-------------|-------------|----------|
| Accuracy     | **80.00%**  | 79.32%      | -0.68 pp |
| Churn Recall | **58%**     | 53%         | -5 pp    |
| Churn F1     | **61%**     | 58%         | -3 pp    |
| ROC-AUC      | **83.58%**  | 83.50%      | -0.08 pp |
'''

In [ ]:
'''Final Observation
Original Logistic Regression:
- Accuracy: 80.00%
- Churn Recall: 58%
- Churn F1: 61%
- ROC-AUC: 83.58%
Clean Feature-Engineered Logistic Regression:
- Accuracy: 79.32%
- Churn Recall: 53%
- Churn F1: 58%
- ROC-AUC: 83.50%'''

In [ ]:
'''Feature Engineering Conclusion
Feature engineering was performed by creating TenureGroup, AvgMonthlySpend, ServiceCount, and IsMonthToMonth. 
Different feature combinations were tested with Logistic Regression, Decision Tree, and Random Forest models.
The experiments showed that the engineered features did not provide a meaningful improvement over the original feature set. 
The original Logistic Regression model achieved 80.00% accuracy, 58% churn recall, 61% churn F1-score, and 83.58% ROC-AUC,
while the cleaned feature-engineered version achieved 79.32% accuracy and 83.50% ROC-AUC.
Therefore, the original feature set will be retained as the baseline for further optimization. 
This experiment also demonstrated that feature engineering should bebased on the actual impact on model performance rather 
than simply adding more features.
'''

In [ ]:
#Hyperparameter Tuning

'''🎯 Goal
Today we will optimize our ML models by testing different hyperparameter combinations.
Flow:
1. Understand Hyperparameters
2. GridSearchCV
3. Tune Logistic Regression
4. Tune Random Forest
5. Compare tuned vs baseline models
6. Select the best configuration
7. Detailed evaluation
8. Save results for the final model'''

In [ ]:
#Import GridSearchCV
# Import GridSearchCV for hyperparameter tuning
from sklearn.model_selection import GridSearchCV

In [ ]:
### Logistic Regression Hyperparameter Tuning

'''We use GridSearchCV to test multiple hyperparameter combinations for Logistic Regression.

The main hyperparameter considered here is C, which controls the strength of regularization.

A smaller C applies stronger regularization, while a larger C allows the model to fit the training data more closely.

ROC-AUC is used as the scoring metric because the dataset is imbalanced.'''

In [ ]:
# Define the hyperparameter grid for Logistic Regression
param_grid_lr = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"]
}

param_grid_lr

In [ ]:
# Create and run GridSearchCV for Logistic Regression
lr_grid = GridSearchCV(
    estimator=LogisticRegression(max_iter=5000),
    param_grid=param_grid_lr,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

lr_grid.fit(X_train_encoded, y_train)

print("Best Parameters:", lr_grid.best_params_)
print("Best Cross-Validation ROC-AUC:", lr_grid.best_score_)

In [ ]:
#Evaluate Tuned Logistic Regression
# Get the best Logistic Regression model
best_lr = lr_grid.best_estimator_

# Make predictions on the test set
y_pred_best_lr = best_lr.predict(X_test_encoded)
y_prob_best_lr = best_lr.predict_proba(X_test_encoded)[:, 1]

# Display test accuracy
print("Test Accuracy:", best_lr.score(X_test_encoded, y_test))

# Display classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_lr))

# Display confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_lr))

# Display ROC-AUC
print("\nROC-AUC:", roc_auc_score(y_test, y_prob_best_lr))

In [ ]:
'''Baseline Logistic Regression ROC-AUC: 0.8358
Tuned Logistic Regression ROC-AUC: 0.8351
Baseline Accuracy: 80.00%
Tuned Accuracy: 80.03%'''

In [ ]:
#Store Tuned Logistic Regression Results
# Store the tuned Logistic Regression results
tuned_lr_results = {
    "Model": "Tuned Logistic Regression",
    "Accuracy": best_lr.score(X_test_encoded, y_test),
    "Churn Recall": classification_report(
        y_test, y_pred_best_lr, output_dict=True
    )["1"]["recall"],
    "Churn F1": classification_report(
        y_test, y_pred_best_lr, output_dict=True
    )["1"]["f1-score"],
    "ROC-AUC": roc_auc_score(y_test, y_prob_best_lr)
}

tuned_lr_results

In [ ]:
#Define Random Forest Hyperparameter Grid
# Define the hyperparameter grid for Random Forest
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5]
}

param_grid_rf

In [ ]:
# Create and run GridSearchCV for Random Forest
rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

rf_grid.fit(X_train_encoded, y_train)

print("Best Parameters:", rf_grid.best_params_)
print("Best Cross-Validation ROC-AUC:", rf_grid.best_score_)

In [ ]:
#Check Best Parameters
# Display the best Random Forest configuration
print("Best Parameters:", rf_grid.best_params_)
print("Best Cross-Validation ROC-AUC:", rf_grid.best_score_)

In [ ]:
#Evaluate Tuned Random Forest
# Get the best Random Forest model
best_rf = rf_grid.best_estimator_

# Make predictions on the test set
y_pred_best_rf = best_rf.predict(X_test_encoded)
y_prob_best_rf = best_rf.predict_proba(X_test_encoded)[:, 1]

# Display test accuracy
print("Test Accuracy:", best_rf.score(X_test_encoded, y_test))

# Display classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_rf))

# Display confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_rf))

# Display ROC-AUC
print("\nROC-AUC:", roc_auc_score(y_test, y_prob_best_rf))

In [ ]:
'''Baseline RF ROC-AUC: 0.8100
Tuned RF ROC-AUC: 0.8330
Baseline RF Accuracy: 79.25%
Tuned RF Accuracy: 78.75%'''

In [ ]:
#Store Tuned Random Forest Results
# Store the tuned Random Forest results
tuned_rf_results = {
    "Model": "Tuned Random Forest",
    "Accuracy": best_rf.score(X_test_encoded, y_test),
    "Churn Recall": classification_report(
        y_test, y_pred_best_rf, output_dict=True
    )["1"]["recall"],
    "Churn F1": classification_report(
        y_test, y_pred_best_rf, output_dict=True
    )["1"]["f1-score"],
    "ROC-AUC": roc_auc_score(y_test, y_prob_best_rf)
}

tuned_rf_results

In [ ]:
'''
Baseline vs Tuned Model Comparison

### Baseline vs Tuned Model Comparison

The baseline and tuned models are compared using Accuracy, Churn Recall,
Churn F1-score, and ROC-AUC.

This comparison helps determine whether hyperparameter tuning provided
a meaningful improvement on the unseen test set.
'''
# Create a comparison table for baseline and tuned models
tuning_comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Type": "Baseline",
        "Accuracy": 0.8000,
        "Churn Recall": 0.58,
        "Churn F1": 0.61,
        "ROC-AUC": 0.8358
    },
    {
        "Model": "Logistic Regression",
        "Type": "Tuned",
        "Accuracy": tuned_lr_results["Accuracy"],
        "Churn Recall": tuned_lr_results["Churn Recall"],
        "Churn F1": tuned_lr_results["Churn F1"],
        "ROC-AUC": tuned_lr_results["ROC-AUC"]
    },
    {
        "Model": "Random Forest",
        "Type": "Baseline",
        "Accuracy": 0.7925,
        "Churn Recall": 0.51,
        "Churn F1": 0.57,
        "ROC-AUC": 0.81
    },
    {
        "Model": "Random Forest",
        "Type": "Tuned",
        "Accuracy": tuned_rf_results["Accuracy"],
        "Churn Recall": tuned_rf_results["Churn Recall"],
        "Churn F1": tuned_rf_results["Churn F1"],
        "ROC-AUC": tuned_rf_results["ROC-AUC"]
    }
])

tuning_comparison

In [ ]:
#Save tuning results
# Save the hyperparameter tuning comparison results
tuning_comparison.to_csv("hyperparameter_tuning_results.csv", index=False)

print("Hyperparameter tuning results saved successfully.")

In [ ]:
### Hyperparameter Tuning Conclusion

'''Hyperparameter tuning was performed using GridSearchCV with 5-fold cross-validation for Logistic Regression and Random Forest.

For Logistic Regression, the best configuration was C=10 with the lbfgs solver. The tuned model achieved 80.03% test accuracy and 83.51% ROC-AUC, which was very similar to the baseline model.

For Random Forest, the best configuration was n_estimators=100, max_depth=5, and min_samples_split=5. The tuned model achieved 78.75% test accuracy and 83.30% ROC-AUC. 

Although ROC-AUC improved compared with the baseline Random Forest, churn recall and F1-score decreased.

Overall, hyperparameter tuning did not provide a significant improvement in overall test-set performance. Therefore, the original Logistic Regression model will be retained as the baseline candidate for further model selection and final evaluation.

— Hyperparameter Tuning: Completed '''

In [ ]:
'''Final model select karna
Final model ko train karna
Accuracy + Precision + Recall + F1
Confusion Matrix
ROC-AUC
ROC Curve
Feature importance / coefficients
Final model conclusion'''

In [ ]:
'''### Final Model Selection

Based on the baseline and hyperparameter tuning experiments, Logistic Regression
is retained as the final model candidate.

The baseline Logistic Regression achieved 80.00% test accuracy and 83.58% ROC-AUC.
Hyperparameter tuning did not provide a meaningful improvement on the unseen test set.

Therefore, the baseline Logistic Regression model will be used for the final evaluation.'''

In [ ]:
# Select Logistic Regression as the final model
final_model = model

print("Final Model:", final_model)

In [ ]:
### Train the Final Model

'''The selected Logistic Regression model is trained on the complete training dataset
using the preprocessed and encoded features.

This trained model will be used for the final evaluation on the unseen test set.'''

In [ ]:
# Train the final Logistic Regression model
final_model.fit(X_train_encoded, y_train)

print("Final model training completed successfully.")

In [ ]:
### Generate Final Predictions

'''The trained final model is used to generate predictions for the unseen test dataset.

Both class predictions and churn probabilities are generated.
The probabilities will be used later for ROC-AUC and ROC curve analysis.'''

In [ ]:
# Generate predictions and churn probabilities
final_predictions = final_model.predict(X_test_encoded)
final_probabilities = final_model.predict_proba(X_test_encoded)[:, 1]

print("Predictions generated successfully.")
print("Number of predictions:", len(final_predictions))

In [ ]:
'''### Final Model Evaluation

The final Logistic Regression model is evaluated using Accuracy, Precision,
Recall, F1-score, and ROC-AUC.

These metrics provide a detailed view of the model's classification performance,
especially for identifying customers who are likely to churn.'''

In [ ]:
# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Calculate final model metrics
final_accuracy = accuracy_score(y_test, final_predictions)
final_precision = precision_score(y_test, final_predictions)
final_recall = recall_score(y_test, final_predictions)
final_f1 = f1_score(y_test, final_predictions)
final_roc_auc = roc_auc_score(y_test, final_probabilities)

# Display the results
print("Final Model Evaluation")
print("----------------------")
print("Accuracy :", final_accuracy)
print("Precision:", final_precision)
print("Recall   :", final_recall)
print("F1-Score :", final_f1)
print("ROC-AUC  :", final_roc_auc)

In [ ]:
'''### Confusion Matrix

The confusion matrix shows the number of true positives, true negatives,
false positives, and false negatives produced by the final model.

It helps identify the types of classification errors made by the model.'''

In [ ]:
# Import confusion matrix functions
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Calculate the confusion matrix
cm = confusion_matrix(y_test, final_predictions)

# Display the confusion matrix
print("Confusion Matrix:")
print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Churn", "Churn"]
).plot()

In [ ]:
'''### ROC Curve

The ROC curve shows the relationship between the True Positive Rate
and False Positive Rate at different classification thresholds.

The Area Under the ROC Curve (ROC-AUC) summarizes the model's ability
to distinguish between churn and non-churn customers.'''

In [ ]:
# Import ROC curve and AUC functions
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Calculate ROC curve values
fpr, tpr, thresholds = roc_curve(y_test, final_probabilities)

# Calculate AUC
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Final Logistic Regression Model")
plt.legend()
plt.show()

In [ ]:
'''### Logistic Regression Feature Coefficients

Logistic Regression uses coefficients to determine the influence of each feature
on the prediction.

Positive coefficients increase the likelihood of churn, while negative
coefficients decrease the likelihood of churn.

The encoded feature names are extracted from the preprocessing pipeline
to interpret the model coefficients.'''

In [ ]:
# Get feature names after one-hot encoding
feature_names = preprocessor.get_feature_names_out()

# Create a DataFrame containing feature names and their coefficients
coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": final_model.coef_[0]
})

# Sort features by coefficient
coefficients_sorted = coefficients.sort_values(
    by="Coefficient",
    ascending=False
)

# Display the top 10 positive and negative coefficients
print("Top 10 Features Increasing Churn:")
display(coefficients_sorted.head(10))

print("\nTop 10 Features Decreasing Churn:")
display(coefficients_sorted.tail(10))

In [ ]:
### Top Features Associated with Higher Churn Probability

'''Positive Logistic Regression coefficients indicate features that increase
the model's predicted log-odds of churn, while negative coefficients indicate
a decrease in the predicted log-odds of churn.

The features with the largest positive coefficients are examined to identify
patterns associated with higher predicted churn probability.'''

In [ ]:
# Display the top 10 features with positive coefficients
print("Top 10 Features Increasing Churn Probability:")

display(
    coefficients_sorted
    .sort_values(by="Coefficient", ascending=False)
    .head(10)
)

In [ ]:
''' Final Model & Detailed Evaluation Conclusion

The final Logistic Regression model was trained and evaluated on the unseen test set.

The model achieved 80.38% accuracy, 64.67% precision, 57.75% recall,
61.02% F1-score, and 83.59% ROC-AUC.

The confusion matrix and ROC curve were used to evaluate the model's
classification performance and its ability to distinguish between churn
and non-churn customers.

Feature coefficient analysis provided interpretability by identifying
encoded features associated with higher or lower predicted churn risk.
These coefficients represent model associations and should not be
interpreted as causal relationships.

Overall, Logistic Regression provides a solid and interpretable baseline
for the customer churn prediction task.

Day 11 — Final Model & Detailed Evaluation: Completed.'''

In [ ]:
#Day 12 — Model Saving & New Customer Prediction
'''1. Save the Final Model
   - Save the trained Logistic Regression model using joblib.
   - File: final_model.pkl
2. Save the Preprocessor
   - Save the fitted preprocessing pipeline.
   - File: preprocessor.pkl
3. Load the Saved Model and Preprocessor
   - Verify that both files can be loaded successfully.
4. Create New Customer Input
   - Define sample customer details using the original feature columns.
5. Preprocess New Customer Data
   - Apply the same preprocessing used during model training.
6. Generate Churn Prediction
   - 0 → No Churn
   - 1 → Churn
7. Generate Churn Probability
   - Calculate the probability of the customer churning.
8. Create a Prediction Function
   - Build a reusable function for predicting churn for new customers.
9. Test the Prediction Pipeline
   - Verify the complete flow:
     Input → Preprocessing → Model → Prediction → Probability
10. Day 12 Conclusion
11. Git Commit & Push
- Upload the Day 12 changes to GitHub.'''

In [ ]:
### 1 Save the Final Model

'''The trained Logistic Regression model is saved using Joblib so that
it can be reused later without retraining the model.

The saved model will be used during prediction and deployment.'''

In [ ]:
import joblib

# Save the trained final model
joblib.dump(final_model, "final_model.pkl")

print("Final model saved successfully.")

In [ ]:
### Save the Preprocessor
'''
The fitted preprocessor is saved using Joblib so that new customer data
can be transformed using the same preprocessing steps applied during training.

This ensures consistency between training data and new prediction data.'''

In [ ]:
# Save the fitted preprocessor
joblib.dump(preprocessor, "preprocessor.pkl")

print("Preprocessor saved successfully.")

In [ ]:
### Load the Saved Model and Preprocessor

'''The saved model and preprocessor are loaded from their files to verify
that they can be successfully reused without retraining.

This step confirms that the saved artifacts are ready for future predictions
and deployment.'''

In [ ]:
# Load the saved model and preprocessor
loaded_model = joblib.load("final_model.pkl")
loaded_preprocessor = joblib.load("preprocessor.pkl")

print("Model and preprocessor loaded successfully.")
print("Loaded Model:", loaded_model)

In [ ]:
'''### Create New Customer Input

A sample customer is created using the original input features required by
the trained model.

The customer data is kept in its original format so that the saved
preprocessor can transform it before prediction.'''

In [ ]:
# Create a sample new customer
new_customer = pd.DataFrame([{
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 5,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 85.50,
    "TotalCharges": 427.50
}])

print("New customer created successfully.")
display(new_customer)

In [ ]:
# Transform the new customer data using the saved preprocessor
new_customer_encoded = loaded_preprocessor.transform(new_customer)

print("New customer data preprocessed successfully.")
print("Encoded shape:", new_customer_encoded.shape)

In [ ]:
'''### Generate Churn Prediction

The preprocessed customer data is passed to the trained Logistic Regression
model to predict whether the customer is likely to churn.

The model returns:
0 → No Churn
1 → Churn'''

In [ ]:
# Generate churn prediction
prediction = loaded_model.predict(new_customer_encoded)

print("Prediction:", prediction[0])

if prediction[0] == 1:
    print("Customer is predicted to churn.")
else:
    print("Customer is predicted to stay.")

In [ ]:
### Generate Churn Probability

'''The model probability represents the estimated likelihood that the customer
belongs to the churn class.

The probability is obtained using the trained Logistic Regression model's
predict_proba() method.'''

In [ ]:
# Calculate churn probability
churn_probability = loaded_model.predict_proba(new_customer_encoded)[0][1]

print(f"Churn Probability: {churn_probability:.2%}")

In [ ]:
'''### Create a Reusable Prediction Function

A reusable prediction function is created to simplify the churn prediction
process for new customers.

The function applies the saved preprocessor, generates the churn prediction,
and calculates the churn probability.'''

In [ ]:
def predict_customer_churn(customer_data):
    # Preprocess the customer data
    customer_encoded = loaded_preprocessor.transform(customer_data)

    # Generate prediction
    prediction = loaded_model.predict(customer_encoded)[0]

    # Generate churn probability
    probability = loaded_model.predict_proba(customer_encoded)[0][1]

    if prediction == 1:
        result = "Churn"
    else:
        result = "No Churn"

    return result, probability


# Test the prediction function
result, probability = predict_customer_churn(new_customer)

print("Prediction:", result)
print(f"Churn Probability: {probability:.2%}")

In [ ]:
'''### Test the Prediction Pipeline

The complete prediction pipeline is tested using a new customer.

The pipeline takes raw customer data, applies the saved preprocessing steps,
generates a prediction, and returns the corresponding churn probability.'''

In [ ]:
# Test the complete prediction pipeline
test_result, test_probability = predict_customer_churn(new_customer)

print("----- Customer Churn Prediction -----")
print("Prediction:", test_result)
print(f"Churn Probability: {test_probability:.2%}")
print("--------------------------------------")

In [ ]:
'''### Day 12 — Model Saving & New Customer Prediction Conclusion

The final Logistic Regression model and preprocessing pipeline were saved
using Joblib and successfully loaded for future use.

A new customer input was processed using the saved preprocessor and passed
to the saved model for prediction.

The model predicted that the sample customer would churn with a predicted
churn probability of 77.58%.

A reusable prediction function was also created to automate the complete
prediction pipeline.

Day 12 — Model Saving & New Customer Prediction: Completed.'''

In [ ]:
'''Day 13 Flow
Deployment Concept
Understand how the trained ML model can be used by an application.
Create Prediction Script
Load final_model.pkl
Load preprocessor.pkl
Accept customer input
Return churn prediction and probability.
Create Streamlit App
Build a simple web interface for customer details.
User enters customer information through UI.
Connect UI with ML Model
Input → Preprocessing → Model → Prediction.
Display Prediction
Show:
Churn / No Churn
Churn Probability
Run the Application Locally
Test the complete application in the browser.
Handle Input Consistency
Ensure the Streamlit inputs match the model's expected features.
Deployment Documentation
Add basic instructions for running the application.
Day 13 Conclusion
Git Commit & Push
Architecture

User Input
↓
Streamlit UI
↓
Preprocessor
↓
Saved Logistic Regression Model
↓
Prediction + Churn Probability'''

In [ ]:
'''### Machine Learning Model Deployment

Model deployment is the process of making a trained machine learning model
available for real-world use.

In this project, the trained Logistic Regression model and preprocessing
pipeline will be integrated into a Streamlit web application.

The application will accept customer information, process the input using
the saved preprocessor, and generate a churn prediction with its probability.'''

In [ ]:
#project Architecture
'''
User Input
    ↓
Streamlit Web Interface
    ↓
Saved Preprocessor
    ↓
Saved Logistic Regression Model
    ↓
Churn Prediction
    ↓
Churn Probability
'''